# Single-GPU work on Colab

## Read this first

**A Colab runtime exposing one GPU cannot run the multi-GPU labs.**

This is the first thing to verify. Assigned hardware, availability, usage
limits and idle policies vary by account and demand; inspect the runtime
instead of assuming a particular GPU model or session duration.

For the multi-GPU labs use `notebooks/kaggle_2xT4_tier2.ipynb` when Kaggle
offers your account a two-GPU accelerator, or use another two-GPU host.

## What this notebook is actually good for

- **Lab 1B memory measurement.** `torch.cuda.max_memory_allocated` is a real
  number. The CPU fallback is process RSS, which includes the interpreter and
  is only good for trends.
- **Single-GPU profiling and kernel timing.** Where your FLOPs actually go
  before any parallelism enters the picture.
- **MFU on one device.** The ceiling every distributed configuration is a
  fraction of. Lab 4 needs it.
- **Any Tier-0 or Tier-1 test**, when your laptop is unavailable. They run on
  CPU and do not need the GPU at all.

Choose a GPU runtime if one is currently available, then run the cells.


## 1. Setup


In [ ]:
!nvidia-smi

# One device. That is expected and it is the point of the warning above.


In [ ]:
import pathlib
import subprocess

REPO_URL = ''  # Paste this repository's public HTTPS clone URL.
if not REPO_URL.startswith('https://'):
    raise ValueError('Set REPO_URL to the public HTTPS Git clone URL first')

checkout = pathlib.Path('/content/dtfs')
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(checkout)], check=True)
%cd /content/dtfs
# Preserve the runtime's CUDA-enabled torch build. Install this package
# without dependencies, then install and validate every non-Torch requirement.
!pip install -q -e . --no-deps
!python -m pip install -q 'numpy>=1.26' 'pyyaml>=6.0' 'matplotlib>=3.8' 'psutil>=5.9' 'setuptools>=83.0' 'pytest>=8.0' 'pytest-timeout>=2.3'
!python -m pip check


In [ ]:
!python scripts/doctor.py

# Expect:
#   Tier 1 (gloo/CPU multi-process):  AVAILABLE
#   Tier 2 (NCCL multi-GPU):          UNAVAILABLE
#
# The second line is correct, not a problem to solve. One GPU cannot host two
# NCCL ranks.


## 2. Tier 0 and Tier 1

These need no GPU. Useful if you are here because your laptop is not. The
command validates the completed published answers in an isolated temporary
copy; use `python tasks.py test LAB=N` when testing your own starter edits.


In [ ]:
!python scripts/test_answers.py -m 'not tier2' -q --timeout=300


## 3. The memory ledger, measured

Lab 1B's central exercise. On a real device the peak-memory numbers mean
something, and the gap between predicted and measured is the deliverable.


In [ ]:
import torch

from common import memory, reference_gpt, repo_env

repo_env.apply()

device = torch.device('cuda')
model = reference_gpt.make_model('small').to(device)
optimizer = model.configure_optimizer(lr=1e-3)

memory.reset_peak()

x = torch.randint(0, model.config.vocab_size, (8, 128), device=device)
y = torch.randint(0, model.config.vocab_size, (8, 128), device=device)

for _ in range(3):
    _logits, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()   # Adam allocates m and v here, not before

ledger = memory.analytic_ledger(model.num_params(), 'fp32', 'adamw', 0, 1)
print(memory.reconcile(ledger, label='small preset, fp32, single GPU'))


### Read the gap

The measured peak sits above the analytic states total, and the difference is
activations, the autograd graph holding saved tensors until backward finishes,
cuBLAS workspaces, and the allocator rounding block sizes.

Attribute it. `memory.activation_estimate` gives you the first term, and what
is left after subtracting it is the framework's overhead on this model at this
batch size. That number is worth knowing before you plan a run that has to fit
in a fixed amount of VRAM.


In [ ]:
act = memory.activation_estimate(
    batch_size=8, seq_len=128,
    hidden=model.config.n_embd,
    n_layers=model.config.n_layer,
    n_heads=model.config.n_head,
    dtype='fp32',
)
measured = memory.measured_peak()

print(f'analytic states      {memory.format_bytes(ledger.total):>12}')
print(f'activation estimate  {memory.format_bytes(act):>12}')
print(f'measured peak        {memory.format_bytes(measured["allocated"]):>12}')
print(f'unexplained          {memory.format_bytes(measured["allocated"] - ledger.total - act):>12}')


## 4. Where the batch size stops fitting

Predict first, then measure. Lab 1B's `predict_fits` is the prediction and
this loop is the measurement.

This deliberately runs until it OOMs. The exception is caught, all tensors and
gradients from the attempt are released, and the allocator cache is emptied before
the next size.


In [ ]:
import gc

results = []
for batch in (4, 8, 16, 32, 64, 128):
    model.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()
    memory.reset_peak()
    oom = False
    try:
        x = torch.randint(0, model.config.vocab_size, (batch, 128), device=device)
        y = torch.randint(0, model.config.vocab_size, (batch, 128), device=device)
        _logits, loss = model(x, y)
        loss.backward()
        model.zero_grad(set_to_none=True)
        peak = memory.measured_peak()['allocated']
        results.append((batch, peak))
        print(f'batch {batch:>4}  peak {memory.format_bytes(peak):>10}')
    except torch.cuda.OutOfMemoryError:
        print(f'batch {batch:>4}  OOM')
        oom = True
    finally:
        model.zero_grad(set_to_none=True)
        for name in ('x', 'y', '_logits', 'loss'):
            if name in globals():
                del globals()[name]
        gc.collect()
        torch.cuda.empty_cache()
    if oom:
        break

if len(results) >= 2:
    slope = (results[-1][1] - results[0][1]) / (results[-1][0] - results[0][0])
    print(f'\nmarginal cost per batch element: {memory.format_bytes(slope)}')
    print('That slope is the activation term. The intercept is everything else.')


## 5. MFU on one device

Lab 4 needs this. Model FLOPs per token is about `6 * n_params` for a forward
and backward pass: 2 per parameter per token in the forward, and twice that in
the backward, which computes both the input gradient and the weight gradient.

This is MFU, not HFU: no recomputation is on, so the two are equal. Turn on
activation checkpointing and HFU rises while MFU does not.


In [ ]:
!python labs/lab4_capstone_2d/bench/mfu.py --preset small --steps 30


## 6. A profiler trace

Where the time goes inside one step. `common.timers.trace` warns when output
crosses its configured threshold; it does not truncate the trace. Keep the
active window small because an unbounded profiler run can overwhelm a viewer.

Download the result and open it at <https://ui.perfetto.dev>.


In [ ]:
from common import timers

path = timers.runs_dir('colab') / 'single_gpu_trace.json'
x = torch.randint(0, model.config.vocab_size, (8, 128), device=device)
y = torch.randint(0, model.config.vocab_size, (8, 128), device=device)

with timers.trace(path, active=3, warmup=1) as prof:
    for _ in range(5):
        _logits, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        prof.step()

print(f'open {path} at https://ui.perfetto.dev')


---

## When you are done

Download anything in `runs/`. Colab discards the filesystem when the session
ends.

And for the multi-GPU labs: `notebooks/kaggle_2xT4_tier2.ipynb`.
